In [8]:
import os
import pandas as pd
from pathlib import Path
import subprocess
from pymatgen.io.cif import CifParser
from molSimplify.Informatics.MOF.PBC_functions import overlap_removal, solvent_removal
from molSimplify.Informatics.MOF.MOF_descriptors import get_MOF_descriptors

Packages:
1. molsimp (3.8.20)
2. zeoplusplus
3. os
4. pandas
5. pymatgen
6. 

In [2]:
# First function: Counting number of cifs to process
def cif_number(cif_folder):
    all_files = os.listdir(cif_folder)
    cif_files = [f for f in all_files if f.endswith('.cif')]
    num_cif_files = len(cif_files)
    return num_cif_files

In [14]:
# Second function: Featurization using molSimplify to get RACs

def get_rac(featurization_directory, occupancy_tolerance=0.9, wiggleroom=1.0, depth=3):

    # Set up the paths
    
    cif_folder = os.path.join(featurization_directory, 'cif')
    primitive_folder = os.path.join(featurization_directory, 'primitive')
    overlap_free_folder = os.path.join(featurization_directory, 'no_overlap')
    solvent_free_folder = os.path.join(featurization_directory, 'no_solvent')
    xyz_folder = os.path.join(featurization_directory, 'xyz')
    os.makedirs(primitive_folder, exist_ok=True)
    os.makedirs(overlap_free_folder, exist_ok=True)
    os.makedirs(solvent_free_folder, exist_ok=True)
    os.makedirs(xyz_folder, exist_ok=True)

    featurization_list = []
    # Process each cif files
    for cif_file in os.listdir(cif_folder):
        if not cif_file.lower().endswith('.cif'):
            continue

        cif_path = os.path.join(cif_folder, cif_file)
        primitive_path = os.path.join(primitive_folder, cif_file)
        overlap_free_path = os.path.join(overlap_free_folder, cif_file)
        solvent_free_path = os.path.join(solvent_free_folder, cif_file)
        xyz_path = os.path.join(xyz_folder, cif_file.replace(".cif", ".xyz"))

        try:
            # Convert to primitive cell
            structure = CifParser(str(cif_path), occupancy_tolerance=occupancy_tolerance).get_structures()[0]
            primitive_structure = structure.get_primitive_structure()
            primitive_structure.to(fmt="cif", filename=primitive_path)

            # Remove overlapping atoms
            overlap_removal(primitive_path, overlap_free_path)

            # Remove solvent
            solvent_removal(overlap_free_path, solvent_free_path, wiggle_room=wiggleroom)

            # Compute RACs
            full_names, full_descriptors = get_MOF_descriptors(
                data=solvent_free_path,
                depth=depth,
                path=featurization_directory,
                xyzpath=xyz_path,
                wiggle_room=wiggleroom
            )

            # Add filename to features
            full_names.append('MOFname')
            full_descriptors.append(cif_file)

            # Save all features into list of dictionary
            featurization = dict(zip(full_names, full_descriptors))
            featurization_list.append(featurization)

        except Exception as e:
            print(f"Skipping {cif_file} due to error: {e}")
            continue

    # Save all features
    df = pd.DataFrame(featurization_list)
    return df


In [15]:
featurization_directory = "/Users/sze/Downloads"
cif_folder = os.path.join(featurization_directory, 'cif')
print(f"Number of .cif files: {cif_number(cif_folder)}")
df = get_rac(featurization_directory)
df.to_csv(os.path.join(featurization_directory, 'rac_featurization_frame.csv'), index=False)



Number of .cif files: 1
('cell vectors: ', 'alpha, beta, gamma = 63.75237, 88.32736, 89.99251')
n_components: 1
labels_components: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]
len is 50
('cell vectors: ', 'alpha, beta, gamma = 63.75237, 88.32736, 89.99251')
176 176
